# Scanpy_Embryo-Imp15265942

- preprocessed by DNA pipelines using cellranger v9.0, reference 2024A

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rcParams
import tqdm as notebook_tqdm
from datetime import date
import scrublet as scr
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import scipy
import sys
import os
import re
import harmonypy as hm

#import src.visualization.scBasic as kvis

f"Last execution: {date.today()}"

In [ ]:
pd.set_option("display.max_columns", 50)
%matplotlib inline

## Set input and output directories

In [ ]:
sample = "Embryo-Imp15265942"

In [ ]:
# -- Input datafiles

#counts file
input_h5 = '/lustre/scratch126/cellgen/vento/rs40/from_iRODs/cellranger900_count_49831_' + sample +'_GRCh38-2024-A/filtered_feature_bc_matrix.h5'

#meta file 
#meta = '/nfs/team292/rs40/projects/human_embryo_models/processed_data/meta/kagawa/meta_reformatted.csv'

#scrublet file
scrublet = f'/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/scrublet/{sample}_scrublet.csv'


#human mouse demultiplex
mouse_human = '/lustre/scratch127/cellgen/cellgeni/tickets/tic-3651/results/{sample}/outs/analysis/gem_classification.csv'


In [ ]:
# -- Outputs
output_dir  = f"/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/scanpy/{sample}"
figure_dir  = output_dir

os.makedirs(output_dir, exist_ok=True)

## Scanpy setting

In [ ]:
sc.set_figure_params(dpi = 600)
sc.set_figure_params(dpi_save = 600)
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(scanpy=True, fontsize=9)
sc.settings.figdir = output_dir

## Read and merge with meta data

In [ ]:
#read the data file
adata = sc.read_10x_h5(input_h5)
adata.var_names_make_unique()

In [ ]:
#read scrublet files
scrulet_obs = pd.read_csv(scrublet,
               index_col = 0,
               sep = ',')

# Merge adata with scrublet data
adata.obs = pd.concat([adata.obs, scrulet_obs], axis=1)
adata.obs

#rename index and save the raw counts
adata.obs.index.names = ['cell.ID']

## QC and minimum filters

In [ ]:
# define mitochondrial genes (note that this may start with "mt-" or "MT-" depending on dataset)
adata.var["mt"] = adata.var_names.str.startswith("MT-") 

#define ribosomal genes. this is not necessary for QC, but maybe biologically intresting?
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

#calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata, qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'], 
             jitter=0.4, multi_panel=True)

### filter cells

In [ ]:
min_genes = 1000

sns.displot(adata.obs, x='n_genes_by_counts', kind="kde", bw_adjust = 0.4)

plt.axvline(min_genes,
            color = 'r',
            linestyle = '--')

In [ ]:
sc.pp.filter_genes(adata, min_cells=5)
adata = adata[adata.obs['n_genes_by_counts'] >= min_genes]
adata = adata[adata.obs['total_counts'] <= 10000000]
adata = adata[adata.obs['total_counts'] >= 1000]
adata = adata[adata.obs['pct_counts_mt'] <= 20]

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'], 
             jitter=0.4, multi_panel=True)

In [ ]:
adata.obs["n_genes_group"] = np.where(
    adata.obs["n_genes_by_counts"] > 5000, "high", "low")

## Normalise

In [ ]:
adata.layers['counts'] = adata.X.copy()

#normalise
sc.pp.normalize_total(adata, target_sum = 1e4)

sc.pp.log1p(adata)

## Score cell cycle

In [ ]:
s_genes = ['MCM5','PCNA','TYMS','FEN1','MCM7','MCM4','RRM1','UNG','GINS2','MCM6','CDCA7','DTL','PRIM1',
           'UHRF1','CENPU','HELLS','RFC2','POLR1B','NASP','RAD51AP1','GMNN','WDR76','SLBP','CCNE2','UBR7',
           'POLD3','MSH2','ATAD2','RAD51','RRM2','CDC45','CDC6','EXO1','TIPIN','DSCC1','BLM','CASP8AP2',
           'USP1','CLSPN','POLA1','CHAF1B','MRPL36','E2F8']
g2m_genes = ['HMGB2','CDK1','NUSAP1','UBE2C','BIRC5','TPX2','TOP2A','NDC80','CKS2','NUF2','CKS1B',
             'MKI67','TMPO','CENPF','TACC3','PIMREG','SMC4','CCNB2','CKAP2L','CKAP2','AURKB','BUB1',
             'KIF11','ANP32E','TUBB4B','GTSE1','KIF20B','HJURP','CDCA3','JPT1','CDC20','TTK','CDC25C',
             'KIF2C','RANGAP1','NCAPD2','DLGAP5','CDCA2','CDCA8','ECT2','KIF23','HMMR','AURKA','PSRC1',
             'ANLN','LBR','CKAP5','CENPE','CTCF','NEK2','G2E3','GAS2L3','CBX5','CENPA']
cell_cycle_genes = s_genes + g2m_genes

In [ ]:
cell_cycle_genes = [x for x in cell_cycle_genes if x in adata.var_names]

adata.var.index = adata.var.index.astype(str)
adata.obs.index = adata.obs.index.astype(str)

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

## Selection of HVG

In [ ]:
sc.pp.highly_variable_genes(adata,
                            flavor = 'seurat_v3',
                            n_top_genes = 4000,
                            subset = False)

sc.pl.highly_variable_genes(adata,log = True)

In [ ]:
## Run PCA on HVG only 

#filter highly variable genes, do PCA
bdata = adata[:, adata.var.highly_variable].copy()

#scale the variance, 0 centre, 10 max
sc.pp.scale(bdata,
            max_value = 10)

#compute PCA
sc.tl.pca(bdata,
          n_comps = 100,
          svd_solver = 'arpack')

#fill NaNs with False so that subsetting to HVGs is possible
adata.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata.obsm['X_pca'] = bdata.obsm['X_pca'].copy()
adata.uns['pca'] = bdata.uns['pca'].copy()

adata.varm['PCs'] = np.zeros(shape=(adata.n_vars,
                                    100))
adata.varm['PCs'][adata.var['highly_variable']] = bdata.varm['PCs']

#Tranfer PCA values from bdata to adata
sc.pl.pca_variance_ratio(adata,
                         n_pcs = 50,
                         log = True)

In [ ]:
n_pcs = 20 # based on rough knee point
n_neighbors = 15

#compute leiden clusters
sc.pp.neighbors(adata,
                n_pcs = n_pcs,
                n_neighbors = n_neighbors, random_state=0)

sc.tl.umap(adata, min_dist=0.5, spread=1.0, random_state=0)

sc.tl.leiden(adata, resolution=0.5)

# -- Modify name to UMAP
adata.obsm[f"umap_without_integration"] = adata.obsm['X_umap']

In [ ]:
adata

In [ ]:
#plot for QC
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False

sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))


QC_plot = sc.pl.umap(adata,
           color = ['total_counts',  'n_genes_by_counts', 'pct_counts_mt', 'scrublet_prediction','scrublet_cluster_score', 'leiden','phase', "n_genes_group"],
           wspace = 0.5,
           legend_fontsize = 'x-small',
           ncols = 4,
           alpha = 0.75,
           size = 10,
#            legend_loc = 'on data',
            save="_"+sample+"_QCplot"
          )


QC_plot
plt.savefig(figure_dir +'/'+sample+'_umap_overview.pdf')

## Visulise marker gene expression 

In [ ]:
marker_genes = {
    'pluripotency' : { "POU5F1", "NANOG","SOX2"}, 
    'Naive': {'KLF17','DPPA3','DPPA5', 'DMNT3L'}, 
    'Prime': {'ZIC2', 'OTX2', 'CD24', 'SFRP2'}, #
    'ICM':{"LAMA4", "PRSS3", "FGF1"},
    'EPI':{'KLF17','NODAL', 'PRDM14','DPPA2', 'SUSD2'},
    'TE':{'GATA2', 'GATA3', 'CDX2','YAP1', 'OVOL1'},
   'polar TE':{'CCR7','NR2F2', "CYP19A1", "DLX5", "MUC15"},
    #'amnion':{'GABRP','ISL1'},
    'HYPO': {'GATA6', 'MSX2', 'HNF4A', 'GATA4', 'SOX17', 'RSPO3', 'APOA1', 'PDGFRA'},
    'Stress':{'CDKN1A', 'DDIT3', 'DDIT4', 'HSPA1A', 'BTG2'},
    'cycling': {'CDK1', 'MKI67', 'TOP2A'},
    'NCC': {'BIK','BAK1'},
    'MEF' :{"COL1A1","COL1A2","COL3A1","FN1","THY1","VIM"}
}

In [ ]:
marker_genes = marker_genes


filtered_marker_genes = {}
skipped_genes = {}

for category, genes in marker_genes.items():
    # Find genes that exist in the dataset
    existing_genes = [gene for gene in genes if gene in adata.var_names]

    # Track skipped genes
    skipped_for_category = [gene for gene in genes if gene not in adata.var_names]

    # Only add category if there are existing genes
    if existing_genes:
        filtered_marker_genes[category] = existing_genes

    # Store skipped genes if any
    if skipped_for_category:
        skipped_genes[category] = skipped_for_category


In [ ]:
# Print skipped genes
print("Skipped Genes:")
for category, genes in skipped_genes.items():
    print(f"{category}: {genes}")

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata,
    groupby="leiden",
    var_names=filtered_marker_genes,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = sample + "_marker.pdf")

In [ ]:
mef_like_markers = [
    "COL1A1","COL1A2","COL3A1","FN1","THY1","VIM",
    "PDGFRA","PDGFRB","DCN","LUM","SPARC","TAGLN","ACTA2","SERPINE1",
]

# keep only markers actually present
mef_like_markers = [g for g in mef_like_markers if g in adata.var_names]

sc.tl.score_genes(adata, mef_like_markers, score_name="mef_score")

sc.pl.umap(
    adata,
    color=["mef_score"],
    vmin=0, cmap="RdPu"
)

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "TE" for i in range(0,11)}  # Initialize all as "unknown"

# Assign specific values
for key in []:
    cl_annotation[key] = "doublet"
            
for key in ["11"]:
    cl_annotation[key] = "MEFs"

for key in [ "10", "9"]:
    cl_annotation[key] = "low quality"
    
for key in []:
    cl_annotation[key] = "TE"
    
for key in ["8"]:
    cl_annotation[key] = "EPI"



adata.obs["sample_celltype"] = adata.obs.leiden.map(cl_annotation)

In [ ]:
#visualise the clusters so far
#plot for QC
sc.settings.set_figure_params(dpi = 100)
sc.set_figure_params(figsize=(3, 3))

g=sc.pl.umap(adata, color=["sample_celltype","leiden"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 25,
           alpha = 0.75,
           wspace=0.5,
           save = sample +"_compiled_integrated_final.pdf")

g

In [ ]:
plt.rcParams.update({'font.size': 4})
plt.rcParams['ytick.labelsize'] = 5

FIGSIZE=(3,3.5)
#rcParams['figure.figsize']=FIGSIZE
sc.set_figure_params(scanpy=True, fontsize=9)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
}):
    # Make Axes
    # Number of needed rows and columns (based on the row with the most columns)
    nrow=len(filtered_marker_genes)
    ncol=max([len(vs) for vs in marker_genes.values()])
    fig,axs=plt.subplots(nrow,ncol,figsize=(2.5*ncol,2*nrow))
    # Plot expression for every marker on the corresponding Axes object
    for row_idx,(cell_type,markers) in enumerate(filtered_marker_genes.items()):
        col_idx=0
        for marker in markers:
            ax=axs[row_idx,col_idx]
            sc.pl.umap(adata,color=marker, size = 10, ax=ax,show=False,frameon=False, cmap="RdPu")
            
            # Add cell type as row label - here we simply add it as ylabel of
            # the first Axes object in the row
            if col_idx==0:
                # We disabled axis drawing in UMAP to have plots without background and border
                # so we need to re-enable axis to plot the ylabel
                ax.axis('on')
                ax.set(xlabel=None)
                ax.tick_params(
                    top='off', bottom='off', left='off', right='off', 
                    labelleft='off', labelbottom='off')
                ax.set_ylabel(cell_type+'\n', rotation=90)
                ax.set(frame_on=False)
            col_idx+=1
        # Remove unused column Axes in the current row
        while col_idx<ncol:
            axs[row_idx,col_idx].remove()
            col_idx+=1

# Alignment within the Figure
fig.tight_layout()
plt.savefig(output_dir +'/'+'umaps.pdf')

In [ ]:
adata.obs.to_csv(output_dir+"/" + sample + ".csv")

In [ ]:
adata.obs['scrublet_prediction'] = adata.obs['scrublet_prediction'].astype(str)

In [ ]:
output_dir

In [ ]:
adata.write_h5ad(output_dir+"/" + sample + "_scanpy.h5ad")